# SparkBody LMA Analysis
**Data:** SparkBody TSV export  
**Layout:** `analysis_lab/lma_analysis.ipynb` — TSV is in `../raw_data/`

In [ ]:
# =====================================
# posefireworks 完整分析 notebook 版本
# =====================================

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from pathlib import Path

# =========================
# 1️ 基本設定
# =========================
TSV_PATH = r"C:\Users\user\posefireworks\analysis_lab\raw_data\SparkBodyDaTa V.2 -  Sheet1 (1).tsv"
FFMPEG_EXE = r"C:\Users\user\posefireworks\analysis_lab\ffmpeg.exe"
PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

# =========================
# 2️ 讀取資料
# =========================
try:
    with open(TSV_PATH, encoding="utf-8") as f:
        raw = f.read()

    tsv_lines = [l for l in raw.split("\n") if "\t" in l]

    df = pd.read_csv(
        io.StringIO("\n".join(tsv_lines)),
        sep="\t",
        header=None,
        names=[
            "sessionId","userId","timestamp","mode","activity",
            "shape_n","weight_n","flow_n","kt","baselineReady",
            "lh_x","lh_y","rh_x","rh_y",
            "ls_x","ls_y","rs_x","rs_y","note"
        ]
    )

    df = df[df["baselineReady"].astype(str).str.upper() == "TRUE"].copy()
    print("✅ 數據加載成功")
    print(df.head())
except Exception as e:
    print(f"❌ 數據讀取失敗: {e}")
    df = pd.DataFrame()

# =========================
# 3️ 動畫匯出函數 (單 session) - 升級肩膀與加長版
# =========================
def export_mp4_no_params(sid):
    vsdf = df[df["sessionId"] == sid].copy().reset_index()
    if vsdf.empty:
        print(f"⚠️ 沒有資料: {sid}")
        return

    fig = plt.figure(figsize=(5,5))
    ax = fig.add_subplot(111)
    ax.set_xlim(1,0)
    ax.set_ylim(1,0)
    ax.set_title(f"Session: {sid}", fontsize=12) # 加個標題

    # 加了第三條線：肩膀 (藍色)
    line_l, = ax.plot([], [], "g-o", lw=4, label="Left Arm")
    line_r, = ax.plot([], [], "r-o", lw=4, label="Right Arm")
    line_s, = ax.plot([], [], "b-", lw=4, alpha=0.5, label="Shoulders")

    def update(frame):
        row = vsdf.iloc[frame]
        # 畫左手
        line_l.set_data([row.ls_x, row.lh_x], [row.ls_y, row.lh_y])
        # 畫右手
        line_r.set_data([row.rs_x, row.rh_x], [row.rs_y, row.rh_y])
        # 畫肩膀 (把左肩和右肩連起來)
        line_s.set_data([row.ls_x, row.rs_x], [row.ls_y, row.rs_y])
        
        return line_l, line_r, line_s

    ani = animation.FuncAnimation(
        fig, update, frames=len(vsdf), interval=200, blit=True
    )

    try:
        # 關鍵修改：把 fps 從 30 降到 3。這樣每張圖會停留 0.33 秒，動作才看得清楚
        writer = animation.FFMpegWriter(fps=3)
        writer.executable = FFMPEG_EXE
        save_path = PLOTS_DIR / f"{sid}.mp4"
        ani.save(save_path, writer=writer)
        print(f"🎬 影片成功產出 (有肩膀版): {save_path}")
    except Exception as e:
        print(f"❌ 影片輸出失敗: {e}")
    plt.close(fig)

# =========================
# 4️ 批次匯出全部 session
# =========================
def export_all_sessions():
    if df.empty:
        print("⚠️ 資料為空，無法輸出")
        return
    for sid in df["sessionId"].unique():
        print(f"🎯 產生 session: {sid}")
        export_mp4_no_params(sid)

# =========================
# 5️ 範例執行
# =========================
if not df.empty:
    # 單 session
    target = df["sessionId"].iloc[0]
    export_mp4_no_params(target)

    # 或批次全部 session
    # export_all_sessions()

✅ 數據加載成功
        sessionId                   userId                 timestamp mode  \
4  sess_9ttx61r21  NCNU_User_1773630044390  2026-03-16T03:00:55.128Z    B   
5  sess_9ttx61r21  NCNU_User_1773630044390  2026-03-16T03:00:59.111Z    B   
6  sess_9ttx61r21  NCNU_User_1773630044390  2026-03-16T03:01:07.201Z    B   
7  sess_9ttx61r21  NCNU_User_1773630044390  2026-03-16T03:01:09.593Z    B   
8  sess_9ttx61r21  NCNU_User_1773630044390  2026-03-16T03:01:17.278Z    B   

              activity  shape_n  weight_n    flow_n        kt  baselineReady  \
4         Baseline_End      1.0  0.400924  0.694450  0.517312           True   
5              Victory      1.0  1.000000  0.596109  0.791362           True   
6              Victory      1.0  1.000000  0.247307  0.913443           True   
7  Fireworks_Explosion      1.0  0.537609  0.484466  0.645480           True   
8                Heart      1.0  0.839316  0.862658  0.633796           True   

     lh_x    lh_y    rh_x    rh_y    ls_x    ls